In [0]:
def load_raw_roads():
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructField,StructType,IntegerType,StringType,DoubleType
    source = spark.sql('DESCRIBE EXTERNAL LOCATION trafficprojectexternallocation').select('url').collect()[0][0]
    
    schema = StructType([
        StructField('Road_ID',IntegerType()),

        StructField('Road_Category_Id',IntegerType()),

        StructField('Road_Category',StringType()),

        StructField('Region_ID',IntegerType()),

        StructField('Region_Name',StringType()),

        StructField('Total_Link_Length_Km',DoubleType()),

        StructField('Total_Link_Length_Miles',DoubleType()),

        StructField('All_Motor_Vehicles',DoubleType())
    ])

    df_raw_roads = (
        spark.readStream
             .format('CloudFiles')
             .option('CloudFiles.format','CSV')
             .schema(schema)
             .option('header','True')
             .option('CloudFiles.schemaLocation',f'{source}/CheckPoint/rawRoadsLoad/schemaInfer')
             .load(f'{source}/Landing/raw_roads')
            )
    return df_raw_roads
    

In [0]:
def write_raw_rooads(df):
    source = spark.sql('DESCRIBE EXTERNAL LOCATION trafficprojectexternallocation').select('url').collect()[0][0]
    ws = (
        df.writeStream
          .format('delta')
          .trigger(availableNow = True)
          .outputMode('append')
          .option('CheckPointLocation',f'{source}/Checkpoint/rawRoadsLoad')
          .toTable('traffic_data.bronze.raw_roads')
    )

In [0]:
df = load_raw_roads()
write_raw_rooads(df)

In [0]:
display(spark.sql('select * from traffic_data.bronze.raw_roads'))

In [0]:
display(spark.sql('select count(*) from traffic_data.bronze.raw_roads'))

In [0]:
source = spark.sql('Describe External Location trafficprojectexternallocation').select('url').collect()[0][0]


In [0]:
schema = structType([
    StructField("Record_ID", Integer Type()),
    StructField("Count_point_id", Integer Type()),
    StructField("Direction_of_travel", StringType()),
    StructField("Year", Integer Type()),
    StructField("Count_date", StringType()),
    StructField("hour", Integer Type()),
    StructField("Region_id", Integer Type()),
    StructField("Region_name", StringType()),
    StructField("Local_authority_name", StringType()),
    StructField("Road_name", StringType()),
    StructField("Road_Category_ID", Integer Type()),
    StructField("Start_junction_road_name", StringType()),
    StructField("End_junction_road_name", StringType()),
    StructField("Latitude", DoubleType()),
    StructField("Longitude", DoubleType()),
    StructField("Link_length_km", DoubleType()),
    StructField("Pedal_cycles", Integer Type()),
    StructField("Two_wheeled_motor_vehicles", Integer Type()),
    StructField("Cars_and_taxis", Integer Type()),
    StructField("Buses_and_coaches", Integer Type()),
    StructField("LGV_Type", Integer Type()),
    StructField("HGV_Type", Integer Type()),
    StructField("EV_Car", Integer Type()),
    StructField("EV_Bike", Integer Type())
])

In [0]:
df = (spark.readStream
           .format('CloudFiles')
           .option('CloudFiles.Format','CSV')
           .option('header','True')
           .schema(schema)
           .option('CloudFiles.schemaLocation', f'{source}/CheckPoint/rawRoadsLoad/schemaInfer')
           .load(f"{source}/Landing/raw_roads")
)

In [0]:
from pyspark.sql import functions as F

In [0]:
df_clean = (
    df.select([
        F.col(c).alias(
            c.strip()
            .lower()
            .replace(' ','_')
        ) for c in df.columns
    ])
)

In [0]:
ws = (
    df_clean.writeStream
        .format('delta')
        .option('mergeSchema','True')
        .option('CheckPointLocation',f'{source}/CheckPoint/rawRoadsLoad')
        .trigger(availableNow = True)
        .toTable('traffic_data.bronze.raw_roads')
)

In [0]:
display(spark.sql('select count(*) from traffic_data.bronze.raw_roads'))